# Cruzber — Margin-Weighted OOS Alert Ranking
## Vertex AI Workbench · Colab Enterprise

**Pipeline:** h=4 v4 · KPI Steps 10 / 20 / 30

| Identidad | Propósito |
|-----------|-----------|
| `hugo@deval.work` (ADC workbench) | Vertex AI, proyecto `thequantitativeledger` |
| **`hdeval@mda.isdi.es`** | Leer/escribir datos en `bucket-isdi-mda-online` |

**Data source (read-only):** `gs://bucket-isdi-mda-online/proyecto-troncal/cruzber/`  
**Output:** `gs://bucket-isdi-mda-online/proyecto-troncal/cruzber/outputs/{RUN_TS}/`

| Step | Description |
|------|-------------|
| 10 | Enrich h4 forecast with KPI margin data |
| 20 | Weekly Top-100 alert ranking by EUR-at-risk |
| 30 | Evaluate precision / recall / lift vs baseline |

## 1 · Instalar dependencias

In [ ]:
# Ejecutar sólo la primera vez en el entorno Vertex AI Workbench
%pip install --quiet \
    google-cloud-storage>=2.14.0 \
    google-cloud-bigquery>=3.11.0 \
    db-dtypes>=1.1.0 \
    pandas>=2.0.0 \
    pyarrow>=12.0.0 \
    plotly>=5.18.0 \
    kaleido>=0.2.1

## 2 · Imports y autenticación

In [ ]:
import io, os, json, subprocess, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from google.cloud import storage
from google.oauth2.credentials import Credentials
import google.auth

# ── 1) Credenciales del Workbench (ADC) ──────────────────────────────────────
# Usadas para operaciones Vertex AI en thequantitativeledger.
# En Workbench/Colab Enterprise vienen del metadata server de la VM.
wb_credentials, wb_project = google.auth.default()
print(f"Workbench project : {wb_project}")
print(f"Workbench creds   : {type(wb_credentials).__name__}")

# ── 2) Credenciales de datos: hdeval@mda.isdi.es ─────────────────────────────
# Esta cuenta tiene acceso a bucket-isdi-mda-online (org ISDI).
# Requisito previo — ejecutar UNA VEZ en un terminal del Workbench:
#   gcloud auth login hdeval@mda.isdi.es
DATA_ACCOUNT = "hdeval@mda.isdi.es"


def _get_data_credentials(account: str) -> Credentials:
    """Obtiene credenciales para la cuenta gcloud indicada."""
    accounts_raw = subprocess.run(
        ["gcloud", "auth", "list", "--format=json"],
        capture_output=True, text=True,
    ).stdout
    available = [a["account"] for a in json.loads(accounts_raw or "[]")]

    if account not in available:
        sep = "=" * 60
        print(f"\n{sep}")
        print(f"  ⚠  ACCIÓN REQUERIDA")
        print(f"  La cuenta '{account}' no está autenticada.")
        print(f"  Abre un terminal en este Workbench y ejecuta:")
        print(f"    gcloud auth login {account}")
        print(f"  Luego vuelve a ejecutar esta celda.")
        print(f"{sep}\n")
        raise RuntimeError(f"Cuenta '{account}' no autenticada en gcloud.")

    token = subprocess.run(
        ["gcloud", "auth", "print-access-token", f"--account={account}"],
        capture_output=True, text=True, check=True,
    ).stdout.strip()
    return Credentials(token=token)


data_credentials = _get_data_credentials(DATA_ACCOUNT)
gcs_data = storage.Client(credentials=data_credentials)
print(f"\nData GCS client   : OK (cuenta: {DATA_ACCOUNT})")
print("Listo para leer gs://bucket-isdi-mda-online/...")

## 3 · Configuración del bucket y rutas

In [ ]:
import datetime

# ── Bucket de datos (accedido con hdeval@mda.isdi.es) ────────────────────────
BUCKET_NAME   = "bucket-isdi-mda-online"
GCS_PREFIX    = "proyecto-troncal/cruzber"
FORECAST_FILE = "forecast_h4_v4"      # se busca .parquet primero, luego .csv
KPI_FILE      = "kpi_por_articulo"    # idem

# ── Run timestamp y prefijo de salida ────────────────────────────────────────
RUN_TS      = datetime.datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")
OUTPUT_PRE  = f"{GCS_PREFIX}/outputs/{RUN_TS}"

print(f"Source  : gs://{BUCKET_NAME}/{GCS_PREFIX}/")
print(f"Output  : gs://{BUCKET_NAME}/{OUTPUT_PRE}/")
print(f"Run TS  : {RUN_TS}")

# ── Objeto bucket con credenciales hdeval@mda.isdi.es ────────────────────────
bucket = gcs_data.bucket(BUCKET_NAME)

# Listar archivos disponibles para verificar acceso
blobs = list(bucket.list_blobs(prefix=GCS_PREFIX, max_results=30))
print(f"\nArchivos en gs://{BUCKET_NAME}/{GCS_PREFIX}/:")
for b in blobs:
    print(f"  {b.name}  ({b.size:,} bytes)")

## 4 · Carga de datos desde GCS

In [ ]:
def read_gcs_df(bucket_obj, prefix: str, stem: str) -> pd.DataFrame:
    """
    Intenta leer un DataFrame desde GCS buscando primero .parquet y luego .csv.
    También acepta archivos con sufijos de fecha como forecast_h4_v4_20260221.parquet
    """
    blobs = list(bucket_obj.list_blobs(prefix=prefix))
    # Prioridad: parquet exacto > parquet con sufijo > csv exacto > csv con sufijo
    candidates = [b.name for b in blobs if stem in b.name.split("/")[-1]]
    parquet = [c for c in candidates if c.endswith(".parquet")]
    csv     = [c for c in candidates if c.endswith(".csv")]

    path = next(iter(parquet or csv), None)
    if path is None:
        raise FileNotFoundError(
            f"No se encontró '{stem}' (.parquet/.csv) en gs://{bucket_obj.name}/{prefix}\n"
            f"Archivos disponibles: {[b.name for b in blobs]}"
        )

    blob = bucket_obj.blob(path)
    data = blob.download_as_bytes()
    fmt  = "parquet" if path.endswith(".parquet") else "csv"
    print(f"  ✓ Leído: gs://{bucket_obj.name}/{path}  ({len(data):,} bytes, fmt={fmt})")

    if fmt == "parquet":
        return pd.read_parquet(io.BytesIO(data))
    else:
        return pd.read_csv(io.StringIO(data.decode("utf-8")))


print("Cargando datos…")
df_forecast = read_gcs_df(bucket, GCS_PREFIX, FORECAST_FILE)
df_kpi      = read_gcs_df(bucket, GCS_PREFIX, KPI_FILE)

print(f"\nforecast_h4_v4 : {df_forecast.shape[0]:,} filas × {df_forecast.shape[1]} columnas")
print(f"kpi_articulo   : {df_kpi.shape[0]:,} filas × {df_kpi.shape[1]} columnas")

## 5 · Exploración y saneamiento de datos

In [ ]:
# ── Forecast ─────────────────────────────────────────────────────────────────
print("=== FORECAST ===")
print(df_forecast.dtypes)
print(f"\nSplits: {df_forecast['split'].value_counts().to_dict()}")
print(f"Nulos relevantes:")
for col in ["p_oos_h4", "q90_h4", "yhat_p50_h4", "y_true_h4", "stockout_event_h4"]:
    n = df_forecast[col].isna().sum() if col in df_forecast.columns else "MISSING"
    print(f"  {col}: {n}")

# Normalizar tipos de fecha
for c in ["decision_week", "target_week"]:
    if c in df_forecast.columns:
        df_forecast[c] = pd.to_datetime(df_forecast[c])

# Normalizar stockout_event a int
if "stockout_event_h4" in df_forecast.columns:
    df_forecast["stockout_event_h4"] = (
        df_forecast["stockout_event_h4"]
        .map(lambda x: 1 if str(x).lower() in ("true","1","yes") else 0)
        .astype("Int64")
    )

print("\n=== KPI ===")
print(df_kpi.dtypes)
for col in ["sku_id","margen_unit","precio_unit_net","tipo_abc"]:
    n = df_kpi[col].isna().sum() if col in df_kpi.columns else "MISSING"
    print(f"  {col}: {n} nulos")

# Deduplicar KPI por sku_id (queda el más reciente)
if "snapshot_ts" in df_kpi.columns:
    df_kpi = (df_kpi
              .sort_values("snapshot_ts", ascending=False)
              .drop_duplicates(subset=["sku_id"])
              .reset_index(drop=True))
else:
    df_kpi = df_kpi.drop_duplicates(subset=["sku_id"]).reset_index(drop=True)

print(f"\nKPI SKUs únicos: {df_kpi['sku_id'].nunique():,}")

## 6 · Step 10 — Enriquecer forecast con economía unitaria de margen

Replica `kpi/10_enrich_forecast_with_kpi.sql`.  
Fórmula central:

$$\text{eur\_at\_risk} = p\_oos_{h4} \times q90_{h4} \times \text{margen\_unit\_clamped}$$

`margen_unit_clamped = max(margen_unit_raw, 0)` — garante no usar márgenes negativos como castigo en el ranking.

In [ ]:
# Columnas KPI a transferir al forecast enriquecido
KPI_COLS = [
    "sku_id",
    "descripcion_articulo", "codigo_familia", "codigo_subfamilia",
    "area_competencia_lc", "tipo_abc",
    "estado_articulo", "obsoleto", "sku_active",
    "primera_venta", "ultima_venta", "dias_en_catalogo",
    "precio_unit_net", "margen_unit_raw", "margen_pct",
]
# Añadir snapshot_ts si existe
if "snapshot_ts" in df_kpi.columns:
    KPI_COLS.append("snapshot_ts")

kpi_slim = df_kpi[[c for c in KPI_COLS if c in df_kpi.columns]].copy()

# Calcular margen_unit clamped (≥ 0) dentro del snapshot KPI
if "margen_unit_raw" in kpi_slim.columns:
    kpi_slim["margen_unit"] = kpi_slim["margen_unit_raw"].clip(lower=0)
elif "margen_unit" not in kpi_slim.columns:
    raise ValueError("Columna 'margen_unit' o 'margen_unit_raw' no encontrada en KPI")

# ── JOIN forecast × kpi (LEFT JOIN para no perder SKUs sin KPI) ──────────────
df_kpi_enriched = df_forecast.merge(kpi_slim, on="sku_id", how="left", suffixes=("", "_kpi"))

# ── Calcular eur_at_risk ──────────────────────────────────────────────────────
df_kpi_enriched["eur_at_risk"] = (
    df_kpi_enriched["p_oos_h4"]
    * df_kpi_enriched["q90_h4"]
    * df_kpi_enriched["margen_unit"]
)

# Flag de match KPI
df_kpi_enriched["kpi_matched"] = df_kpi_enriched["margen_unit"].notna().astype(int)

# Estadísticas de match
n_total  = len(df_kpi_enriched)
n_match  = df_kpi_enriched["kpi_matched"].sum()
n_val    = df_kpi_enriched[df_kpi_enriched["split"] == "VAL"]["sku_id"].nunique()
match_pct = 100 * n_match / n_total

print(f"Filas enriquecidas : {n_total:,}")
print(f"KPI match          : {n_match:,}  ({match_pct:.1f}%)")
print(f"SKUs únicos en VAL : {n_val:,}")
print(f"\nEUR at risk (VAL) — estadísticas:")
print(df_kpi_enriched[df_kpi_enriched["split"]=="VAL"]["eur_at_risk"]
      .describe().round(4))

## 7 · Step 20 — Ranking semanal Top-100 por margen en riesgo

Replica `kpi/20_alerts_top100_h4_margin.sql`.  
Criterio de ordenación por semana:
1. `sku_active = 1` primero
2. `eur_at_risk DESC` (primario)
3. `p_oos_h4 DESC` (desempate)
4. `q90_h4 DESC` (desempate secundario)

In [ ]:
# Restricción al split VAL (evaluación)
val = df_kpi_enriched[df_kpi_enriched["split"] == "VAL"].copy()

# Añadir columnas de verdad-terrena estandarizadas
val["true_stockout_label"] = val["stockout_event_h4"].fillna(0).astype(int)
val["true_stockout_sales0"] = (val["y_true_h4"].fillna(0) == 0).astype(int)

# Columna de prioridad: SKUs activos primero
val["_active_sort"] = val["sku_active"].fillna(0).map(lambda x: 0 if x == 1 else 1)

# Ranking por semana de decisión
val_ranked = (
    val
    .sort_values(
        ["decision_week", "_active_sort", "eur_at_risk", "p_oos_h4", "q90_h4"],
        ascending=[True, True, False, False, False]
    )
    .assign(rank_in_week=lambda df: df.groupby("decision_week").cumcount() + 1)
)

# Top-100 por semana
TOP_N = 100
alerts_margin = val_ranked[val_ranked["rank_in_week"] <= TOP_N].copy()
alerts_margin.drop(columns=["_active_sort"], inplace=True)

n_weeks  = alerts_margin["decision_week"].nunique()
n_alerts = len(alerts_margin)
print(f"Semanas de decisión cubiertas : {n_weeks}")
print(f"Alertas totales (Top-{TOP_N})  : {n_alerts:,}  (esperado ≈ {n_weeks * TOP_N:,})")
print(f"\nTop-5 alertas más críticas (por eur_at_risk):")
print(
    alerts_margin[["decision_week","sku_id","descripcion_articulo",
                   "p_oos_h4","q90_h4","margen_unit","eur_at_risk",
                   "true_stockout_label"]]
    .nlargest(5, "eur_at_risk")
    .to_string(index=False)
)

## 8 · Step 30 — Evaluación: precision / recall / lift@100

Replica `kpi/30_eval_alerts_top100_h4_margin_pooled.sql`.

$$\text{lift@100} = \frac{\text{precision@100}}{\text{prevalencia base}}$$

Referencia pipeline Cloud Run: **GLOBAL=11.08×  |  REST=13.12×  |  HIGH\_SEASON=6.32×**

In [ ]:
def evaluate_margin_alerts(alerts_df: pd.DataFrame, universe_df: pd.DataFrame,
                           label_col: str = "true_stockout_label") -> pd.DataFrame:
    """
    Calcula precision@100, recall@100, lift@100 y agregados de negocio
    para cada season_group + GLOBAL.
    """
    rows = []

    groups = list(universe_df["season_group"].unique()) + ["ALL"]

    for sg in groups:
        if sg == "ALL":
            u_mask = slice(None)
            a_mask = slice(None)
            u = universe_df
            a = alerts_df
        else:
            u = universe_df[universe_df["season_group"] == sg]
            a = alerts_df[alerts_df["season_group"] == sg]

        n_universe  = len(u)
        n_stockouts = u[label_col].sum()
        prevalence  = n_stockouts / n_universe if n_universe else 0

        n_alerts    = len(a)
        n_tp        = a[label_col].sum()
        precision   = n_tp / n_alerts if n_alerts else 0
        recall      = n_tp / n_stockouts if n_stockouts else 0
        lift        = precision / prevalence if prevalence else 0

        sum_eur     = a["eur_at_risk"].fillna(0).sum()
        avg_eur     = a["eur_at_risk"].fillna(0).mean()

        rows.append({
            "season_group"          : sg,
            "n_universe"            : n_universe,
            "n_stockouts"           : int(n_stockouts),
            "prevalence"            : round(prevalence, 4),
            "n_alerts"              : n_alerts,
            "n_true_positives"      : int(n_tp),
            "precision_at_100"      : round(precision, 4),
            "recall_at_100"         : round(recall, 4),
            "lift_at_100"           : round(lift, 2),
            "sum_eur_at_risk_top100": round(sum_eur, 2),
            "avg_eur_at_risk_top100": round(avg_eur, 4),
        })

    return pd.DataFrame(rows)


# Evaluar con etiqueta de modelo (stockout_event) y de sales=0
eval_model  = evaluate_margin_alerts(alerts_margin, val, label_col="true_stockout_label")
eval_sales0 = evaluate_margin_alerts(alerts_margin, val, label_col="true_stockout_sales0")

# Tabla final combinada
eval_final = eval_model.merge(
    eval_sales0[["season_group","precision_at_100","recall_at_100","lift_at_100",
                 "n_true_positives","n_stockouts"]],
    on="season_group", suffixes=("_model","_sales0")
)

print("=== EVALUACIÓN DE ALERTAS POR MARGEN (Top-100 por semana) ===\n")
print(eval_final[[
    "season_group","n_universe","prevalence_model",
    "precision_at_100_model","recall_at_100_model","lift_at_100_model",
    "sum_eur_at_risk_top100","avg_eur_at_risk_top100"
]].to_string(index=False))

In [ ]:
# ── Comparación: ranking estándar (policy B: p_oos×q90) vs ranking por margen ─
val["score_policy_b"] = val["p_oos_h4"] * val["q90_h4"]

val_std_ranked = (
    val
    .sort_values(["decision_week","score_policy_b"], ascending=[True, False])
    .assign(rank_std=lambda df: df.groupby("decision_week").cumcount() + 1)
)
alerts_std = val_std_ranked[val_std_ranked["rank_std"] <= TOP_N].copy()

eval_std = evaluate_margin_alerts(alerts_std, val, label_col="true_stockout_label")

# Tabla comparativa
comp = eval_model[["season_group","lift_at_100","sum_eur_at_risk_top100"]].rename(
    columns={"lift_at_100":"lift_margin","sum_eur_at_risk_top100":"eur_margin"}
).merge(
    eval_std[["season_group","lift_at_100","sum_eur_at_risk_top100"]].rename(
        columns={"lift_at_100":"lift_std","sum_eur_at_risk_top100":"eur_std"}
    ),
    on="season_group"
)
comp["delta_lift"]   = (comp["lift_margin"]   - comp["lift_std"]).round(2)
comp["delta_eur"]    = (comp["eur_margin"]    - comp["eur_std"]).round(2)
comp["eur_mejora_%"] = ((comp["delta_eur"] / comp["eur_std"].replace(0, np.nan))*100).round(1)

print("=== Ranking Margen vs Ranking Estándar (Policy B) ===\n")
print(comp.to_string(index=False))

## 9 · Visualizaciones

In [ ]:
# ── Fig 1: Lift@100 Margen vs Estándar por temporada ─────────────────────────
fig1 = go.Figure()
sg_order = [s for s in ["REST","HIGH_SEASON","ALL"] if s in comp["season_group"].values]

fig1.add_bar(
    x=comp["season_group"], y=comp["lift_margin"],
    name="Ranking Margen", marker_color="#1f77b4"
)
fig1.add_bar(
    x=comp["season_group"], y=comp["lift_std"],
    name="Ranking Estándar (Policy B)", marker_color="#aec7e8"
)
fig1.update_layout(
    title="Lift@100 — Margen vs Estándar por temporada",
    yaxis_title="Lift@100",
    barmode="group",
    template="plotly_white",
    height=420,
    legend=dict(orientation="h", yanchor="bottom", y=1.02)
)
fig1.show()

In [ ]:
# ── Fig 2: Euros en riesgo acumulado por semana (perfil estacional) ───────────
weekly_eur = (
    alerts_margin
    .groupby(["decision_week","season_group"], sort=True)
    .agg(sum_eur_at_risk=("eur_at_risk","sum"), n_alerts=("sku_id","count"))
    .reset_index()
)

fig2 = px.bar(
    weekly_eur,
    x="decision_week", y="sum_eur_at_risk",
    color="season_group",
    color_discrete_map={"HIGH_SEASON": "#e74c3c", "REST": "#3498db"},
    title="Euros de Margen en Riesgo (Top-100 alerts, acumulado semanal) — VAL 2024",
    labels={"sum_eur_at_risk": "€ en riesgo", "decision_week": "Semana de decisión"},
    template="plotly_white",
    height=420
)
fig2.update_xaxes(dtick="M1", tickformat="%b %Y")
fig2.show()

In [ ]:
# ── Fig 3: Scatter p_oos vs eur_at_risk (coloreado por tipo_abc) ──────────────
sample = alerts_margin.dropna(subset=["eur_at_risk","p_oos_h4"]).sample(
    min(3000, len(alerts_margin)), random_state=42
)

fig3 = px.scatter(
    sample,
    x="p_oos_h4", y="eur_at_risk",
    color="tipo_abc",
    size="q90_h4",
    size_max=18,
    hover_data=["sku_id","descripcion_articulo","margen_unit","season_group"],
    opacity=0.65,
    title="Probabilidad OOS vs Euros en Riesgo — Top-100 alerts (muestra VAL 2024)",
    labels={"p_oos_h4": "P(OOS) h=4", "eur_at_risk": "€ en riesgo"},
    template="plotly_white",
    height=500
)
fig3.add_vline(x=0.5, line_dash="dot", line_color="gray",
               annotation_text="umbral p=0.50")
fig3.show()

In [ ]:
# ── Fig 4: Top-20 SKUs por euros acumulados en riesgo ─────────────────────────
top20 = (
    alerts_margin
    .groupby(["sku_id","descripcion_articulo","tipo_abc","codigo_familia"], sort=False)
    .agg(
        total_eur_at_risk=("eur_at_risk","sum"),
        n_semanas_en_alerta=("decision_week","nunique"),
        avg_p_oos=("p_oos_h4","mean"),
        avg_margen_unit=("margen_unit","mean"),
        n_stockouts_reales=("true_stockout_label","sum"),
    )
    .reset_index()
    .nlargest(20, "total_eur_at_risk")
)

fig4 = px.bar(
    top20.sort_values("total_eur_at_risk"),
    x="total_eur_at_risk",
    y=top20.sort_values("total_eur_at_risk")["sku_id"].astype(str)
      + " – " + top20.sort_values("total_eur_at_risk")["descripcion_articulo"].fillna(""),
    color="tipo_abc",
    orientation="h",
    title="Top-20 SKUs por Euros Totales en Riesgo (VAL 2024, acumulado)",
    labels={"x": "€ acumulados en riesgo", "y": "SKU"},
    template="plotly_white",
    height=560
)
fig4.update_layout(showlegend=True, yaxis_title="")
fig4.show()

print("\nTabla Top-20:")
print(top20[["sku_id","descripcion_articulo","total_eur_at_risk",
             "n_semanas_en_alerta","avg_p_oos","n_stockouts_reales"]]
      .to_string(index=False))

## 10 · Exportar resultados a GCS

Los artefactos se guardan en `gs://bucket-isdi-mda-online/proyecto-troncal/cruzber/outputs/`

In [ ]:
from datetime import datetime

RUN_TS      = datetime.utcnow().strftime("%Y%m%d_%H%M%S")
OUTPUT_PRE  = f"{GCS_PREFIX}/outputs/{RUN_TS}"


def upload_df_parquet(df: pd.DataFrame, blob_name: str, description: str = "") -> str:
    """Sube un DataFrame como parquet a GCS y devuelve la URI."""
    buf = io.BytesIO()
    df.to_parquet(buf, index=False, engine="pyarrow")
    buf.seek(0)
    blob = bucket.blob(blob_name)
    blob.upload_from_file(buf, content_type="application/octet-stream")
    uri = f"gs://{BUCKET_NAME}/{blob_name}"
    print(f"  ✓ {description or blob_name}  →  {uri}  ({len(df):,} filas)")
    return uri


def upload_df_csv(df: pd.DataFrame, blob_name: str, description: str = "") -> str:
    """Sube un DataFrame como CSV a GCS."""
    buf = io.StringIO()
    df.to_csv(buf, index=False)
    blob = bucket.blob(blob_name)
    blob.upload_from_string(buf.getvalue(), content_type="text/csv")
    uri = f"gs://{BUCKET_NAME}/{blob_name}"
    print(f"  ✓ {description or blob_name}  →  {uri}  ({len(df):,} filas)")
    return uri


def upload_plotly_html(fig, blob_name: str, description: str = "") -> str:
    """Sube un gráfico Plotly como HTML interactivo a GCS."""
    html = fig.to_html(include_plotlyjs="cdn")
    blob = bucket.blob(blob_name)
    blob.upload_from_string(html, content_type="text/html")
    uri = f"gs://{BUCKET_NAME}/{blob_name}"
    print(f"  ✓ {description or blob_name}  →  {uri}")
    return uri


print(f"Exportando a gs://{BUCKET_NAME}/{OUTPUT_PRE}/\n")

# 1. Forecast enriquecido completo (todas las particiones)
upload_df_parquet(
    df_kpi_enriched,
    f"{OUTPUT_PRE}/forecast_h4_v4_kpi.parquet",
    "forecast_h4_v4_kpi (todas particiones)"
)

# 2. Alertas Top-100 rankeadas por margen (VAL)
upload_df_parquet(
    alerts_margin,
    f"{OUTPUT_PRE}/alerts_top100_h4_margin.parquet",
    "alerts_top100_h4_margin"
)

# 3. Tabla de evaluación (precision/recall/lift)
upload_df_csv(
    eval_final,
    f"{OUTPUT_PRE}/eval_summary_margin.csv",
    "eval_summary_margin"
)

# 4. Top-20 SKUs críticos
upload_df_csv(
    top20,
    f"{OUTPUT_PRE}/top20_skus_eur_at_risk.csv",
    "top20_skus_eur_at_risk"
)

# 5. Gráficos interactivos HTML
upload_plotly_html(fig1, f"{OUTPUT_PRE}/fig1_lift_comparison.html", "Fig1 Lift")
upload_plotly_html(fig2, f"{OUTPUT_PRE}/fig2_eur_weekly.html",      "Fig2 € semanal")
upload_plotly_html(fig3, f"{OUTPUT_PRE}/fig3_scatter_poos_eur.html","Fig3 Scatter")
upload_plotly_html(fig4, f"{OUTPUT_PRE}/fig4_top20_skus.html",      "Fig4 Top-20")

print(f"\n✅ Exportación completada en gs://{BUCKET_NAME}/{OUTPUT_PRE}/")